## Upload 20 Random Images to S3 with `laila.guarantee`

Create 20 random RGB images, upload them to an `S3Pool` inside `with laila.guarantee:`, and verify that the uploads completed when the context exits.


In [ ]:
%load_ext autoreload
%autoreload 2

### Objective
Import dependencies, configure paths, and load the S3 pool class used by this example.


In [ ]:
import numpy as np
from PIL import Image

import laila
from laila.pool import S3Pool


### Objective
Define the S3 pool helper and the constants for generating 20 deterministic random images.


In [ ]:
POOL_NICKNAME = "guarantee_random_images_s3_pool"
IMAGE_COUNT = 20
IMAGE_SIZE = (64, 64)

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"


def build_s3_pool() -> S3Pool:
    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
        nickname=POOL_NICKNAME,
    )


### Objective
Create the S3 pool, generate all 20 random images first, then upload the full list inside `with laila.guarantee:` without manually waiting on the returned futures.


In [ ]:
s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=POOL_NICKNAME)

rng = np.random.default_rng(42)
image_entries = []

for i in range(IMAGE_COUNT):
    image_name = f"guarantee_image_{i}"
    image_array = rng.integers(0, 256, size=(*IMAGE_SIZE, 3), dtype=np.uint8)
    image = Image.fromarray(image_array, mode="RGB")
    entry = laila.constant(data=image, nickname=image_name)
    image_entries.append(entry)

with laila.guarantee:
    laila.memorize(image_entries, pool_nickname=POOL_NICKNAME)

print("Uploaded images:", len(image_entries))
print("First image id:", image_entries[0].global_id)
print("Last image id:", image_entries[-1].global_id)


### Objective
Remember a few uploaded images from S3 and verify that they come back as `PIL.Image.Image` objects.


In [ ]:
sample_entries = image_entries[:3]
remember_future = laila.remember(
    [entry.global_id for entry in sample_entries],
    pool_nickname=POOL_NICKNAME,
)
remembered_images = remember_future.wait()

assert len(remembered_images) == len(sample_entries)
assert all(isinstance(entry.data, Image.Image) for entry in remembered_images)

[(entry.global_id, entry.data.size, entry.data.mode) for entry in remembered_images]


### Objective
Clean up the S3 pool so the example does not leave uploaded images behind.


In [ ]:
s3_pool.empty()
print("Emptied source S3 pool:", POOL_NICKNAME)
